# Prompt U-Net Version 320 Training

# v320 — Control Experiment: v21 Architecture + v310-era Data Pipeline

**Research question:** Do the performance differences between v21 and later
models (v292–v313) come **purely from the new training data / preprocessing**?

## What is identical to v21

| Component | v21 value |
|---|---|
| **Filter schedule** | `[32, 64, 128, 256, 512]` + 1024 bottleneck |
| **Decoder** | `Conv2DTranspose` (not bilinear UpSampling) |
| **Prompt fusion** | plain `Add()` — no SE attention |
| **Augmentation** | 10 % probability per stage (photo / geo / morph) |
| **Augmentation ops** | RandomFlip, RandomRotation ±5 %, RandomZoom ±5 %, RandomTranslation ±5 %, RandomBrightness/Contrast/Noise |
| **Loss** | `binary_crossentropy` |
| **Optimizer** | `Adam` + `ExponentialDecay` (LR 1e-3, decay 0.85 / 2 000 ep) |
| **Epochs** | 3 664 |
| **Batch size** | 128 |
| **dp\_training** | 3 500 |
| **offset** | 12 |
| **max\_number\_labels** | 4 |
| **Dataset refresh** | every 75 epochs |
| **Validation loop** | every 300 epochs |

## What is different from v21 (infrastructure only)

| Component | Change |
|---|---|
| **DataGenerator** | Current `DataGenerator.py` (isotropic volumes, label-guided 128×128 patch crop, pure-numpy, fast valid-slice index) |
| **Normalization** | `universal_normalization` (CT hard-coded HU stats, MRI masked z-score with percentile clipping) |
| **Training data** | 3 datasets via `DataLoader_npz`: `nako_combined`, `total_seg_combined`, `msd_combined` |
| **`train_step()`** | Decorated with `@tf.function` (graph mode execution) |
| **`train_epoch()`** | Native `for z in train_dataset` loop (no manual `iter` / `next`) |
| **Pipeline** | Persistent `tf.data` graph (from-generator + augmentation map built once) |

## Setup

In [1]:
import os
import sys
import gc
import math as _m
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import mlflow
import tensorflow as tf

# v320: pure float32 — no mixed precision (matches v21 implicit float32)
# tf.keras.mixed_precision.set_global_policy("mixed_float16")

import logging
tf.get_logger().setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(f"TF  : {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

2026-04-20 15:49:17.650262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776692957.685720      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776692957.697036      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776692957.721510      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776692957.721554      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776692957.721563      55 computation_placer.cc:177] computation placer alr

TF  : 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Allow importing from project root
notebook_dir = Path().resolve()
project_root  = notebook_dir.parent
sys.path.insert(0, str(project_root))

from data.DataLoader_npz import DataLoader_npz
from data.DataGenerator  import DataGenerator

from utils.metrics        import dice_score_tf
from utils.visualization  import plot_result

from prompt_unet_320 import PromptUNet   # v320 architecture

## Data Loading

Three datasets — same as v310–313.

In [3]:
dataset_paths = [
    "data/train_data/nako_combined.npz",
    "data/train_data/total_seg_combined.npz",
    "data/train_data/msd_combined.npz",
]

dataloader    = DataLoader_npz(dataset_paths, val_size=0.01)
datagenerator = DataGenerator(dataloader)

print(f"Image size: {datagenerator.height} x {datagenerator.width}")


Loading NPZ dataset(s)…
Loaded 61 PIDs from /home/dpxuser/prompt-unet/data/train_data/nako_combined.npz
Loaded 45 PIDs from /home/dpxuser/prompt-unet/data/train_data/total_seg_combined.npz
Loaded 40 PIDs from /home/dpxuser/prompt-unet/data/train_data/msd_combined.npz

Final dataset size: 146 patients.

Image size: 128 x 128


## Hyperparameters

Identical to v21.

In [4]:
version           = "p_unet_320"

epochs            = 3664
batch_size        = 128
dp_training       = 3500
dp_testing        = 1000

offset            = 12
max_number_labels = 4

new_ds       = 75    # refresh training data every N epochs (v21 value)
new_val_loop = 300   # run validation every N epochs (v21 value)

## Model & Optimizer

Architecture: v21 filter schedule `[32, 64, 128, 256, 512]` + 1024 bottleneck.

LR schedule: `ExponentialDecay` — identical to v21:
- `initial_learning_rate = 0.001`
- `decay_rate = 0.85` (staircase)
- `decay_steps = steps_per_epoch × 2000`

In [5]:
# ── Build model ───────────────────────────────────────────────────────────────
model = PromptUNet(height=datagenerator.height, width=datagenerator.width)

# Warm-up forward pass to fully initialise all layers
_dummy_x = tf.random.uniform([1, datagenerator.height, datagenerator.width, 1])
_dummy_p = tf.random.uniform([1, datagenerator.height, datagenerator.width, 2])
_ = model.this([_dummy_x, _dummy_p])

print(f"Trainable params: {model.this.count_params():,}")

# ── v21 ExponentialDecay LR schedule ──────────────────────────────────────────
steps_per_epoch = dp_training // batch_size
decay_epochs    = 2000          # reduce every x epochs (v21 value)
decay_steps     = steps_per_epoch * decay_epochs

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=decay_steps,
    decay_rate=0.85,
    staircase=True,
)

model.optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

I0000 00:00:1776693174.744325      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46640 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1776693178.472817      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


Trainable params: 45,628,641


## Persistent tf.data Pipeline

The pipeline graph (including `.map(augmenter)`) is built **once** here.  
When fresh training data is needed, only the numpy buffer is swapped — no TF
graph nodes accumulate over time.

The augmentation function used is the **exact v21 augmentation** (10 % per stage,
same photometric / geometric / morphological ops), accessed via `model.v21_augmentation`.

In [6]:
# ── Shared numpy buffer ───────────────────────────────────────────────────────
_buf = {"x": None, "y": None, "p": None, "m": None}

def refresh_train_data():
    """Pull fresh random training data into the numpy buffer."""
    x_np, y_np, p_np, m_np, _ = datagenerator.get_data_points_numpy(
        max_data_points   = dp_training,
        offset            = offset,
        max_number_labels = max_number_labels,
    )
    _buf["x"] = x_np
    _buf["y"] = y_np
    _buf["p"] = p_np
    _buf["m"] = m_np
    gc.collect()


def _data_gen():
    """Yields one shuffled sample at a time from the numpy buffer."""
    n       = len(_buf["x"])
    indices = np.random.permutation(n)
    for i in indices:
        yield _buf["x"][i], _buf["y"][i], _buf["p"][i], _buf["m"][i]


H, W = datagenerator.height, datagenerator.width

# ── Two-stage augmentation pipeline ──────────────────────────────────────────
#
# Stage 1  v21_augmentation_tf()  — pure TF ops (geo + photo)
#          Called directly as a .map() step in graph mode.
#          RandomRotation / RandomZoom are safe here (no EagerPyFunc wrapper).
#
# Stage 2  v21_augmentation_morph()  — scipy/numpy ops (prompt morphology)
#          Must go through tf.py_function so it runs eagerly.
#
# WHY two stages?
#   Keras's RandomRotation (and Zoom/Translation) have a TF bug where a
#   4-channel tensor processed inside tf.py_function (EagerPyFunc) causes:
#       "Expected begin, end, and strides to be 1D equal size tensors,
#        but got shapes [4], [1], and [1] instead."
#   Moving the geo augmentation to a plain .map() step avoids EagerPyFunc
#   entirely and lets TF trace the ops correctly.

def _augment_stage1(x, y, p, m):
    """Stage 1: geo + photometric augmentation — pure TF, graph-mode safe."""
    x, y, p, m = model.v21_augmentation_tf(x, y, p, m)
    x.set_shape((H, W, 1))
    y.set_shape((H, W, 1))
    p.set_shape((H, W, 2))
    return x, y, p, m


def _augment_stage2(x, y, p, m):
    """Stage 2: prompt morphological augmentation — wrapped in py_function."""
    x, y, p = tf.py_function(
        lambda xi, yi, pi: model.v21_augmentation_morph(xi, yi, pi),
        [x, y, p],
        [tf.float32, tf.float32, tf.float32],
    )
    x.set_shape((H, W, 1))
    y.set_shape((H, W, 1))
    p.set_shape((H, W, 2))
    return x, y, p, m


train_ds = (
    tf.data.Dataset.from_generator(
        _data_gen,
        output_signature=(
            tf.TensorSpec(shape=(H, W, 1), dtype=tf.float32),  # image
            tf.TensorSpec(shape=(H, W, 1), dtype=tf.float32),  # label
            tf.TensorSpec(shape=(H, W, 2), dtype=tf.float32),  # prompt
            tf.TensorSpec(shape=(),        dtype=tf.float32),  # modality
        )
    )
    .map(_augment_stage1, num_parallel_calls=tf.data.AUTOTUNE)
    .map(_augment_stage2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Pipeline ready.")


Pipeline ready.


## Training

Loop structure mirrors v21 exactly:
- Checkpoint every 8 epochs
- Validation every `new_val_loop` (300) epochs
- Dataset refresh every `new_ds` (75) epochs

In [7]:
def fit(epochs):
    mlflow.set_experiment(version)

    with mlflow.start_run():

        mlflow.log_params({
            "batch_size"         : batch_size,
            "max_number_labels"  : max_number_labels,
            "num_epochs"         : epochs,
            "dp_training"        : dp_training,
            "offset"             : offset,
            "loss_function"      : "binary_crossentropy",
            "lr_initial"         : 0.001,
            "lr_decay_rate"      : 0.85,
            "lr_decay_epochs"    : decay_epochs,
            "trainable_params"   : model.this.count_params(),
            "architecture"       : "v21_exact",
            "filter_schedule"    : "[32,64,128,256,512]+1024",
            "decoder"            : "Conv2DTranspose",
            "augmentation"       : "v21_10pct",
            "data_pipeline"      : "DataLoader_npz+DataGenerator_current",
            "normalization"      : "universal_normalization",
            "train_step"         : "graph_mode_tf.function",
            "train_epoch"        : "native_for_loop",
            "datasets"           : "nako+total_seg+msd",
        })

        # ── Validation dataset (built once, no augmentation) ───────────────────
        val_x, val_y, val_p, val_m, _ = datagenerator.get_val_data_points_numpy(
            max_data_points   = dp_testing,
            offset            = offset,
            max_number_labels = max_number_labels,
        )
        test_ds = (
            tf.data.Dataset.from_tensor_slices((val_x, val_y, val_p, val_m))
            .batch(1)
        )

        # ── Prime the training buffer before the loop ──────────────────────────
        refresh_train_data()

        for epoch in range(epochs):

            model.train_loss.reset_state()

            # Log learning rate
            lr = model.optimizer.learning_rate
            if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
                lr = float(lr(epoch * steps_per_epoch))
            else:
                lr = float(lr.numpy())
            mlflow.log_metric("learning_rate", lr, step=epoch)

            # Checkpoint every 8 epochs
            if epoch % 8 == 0 and epoch != 0:
                model.this.save(f"{version}.keras")

            # Validation every new_val_loop epochs
            if epoch % new_val_loop == 0 and epoch != 0:
                total_dice = 0.0
                for z in test_ds:
                    pred = model.this([z[0], z[2]], training=False)
                    total_dice += float(dice_score_tf(z[1][..., 0:1], pred))
                val_loss = 1.0 - total_dice / dp_testing
                mlflow.log_metric("validation_loss", val_loss, step=epoch)
                print(f"  Validation loss: {val_loss:.4f}")

            # Refresh training data every new_ds epochs
            if epoch % new_ds == 0 and epoch != 0:
                # Visualise one validation prediction
                z_test = next(iter(test_ds))
                pred   = model.this([z_test[0], z_test[2]], training=False)
                plot_result(z_test[0][0], z_test[1][0], z_test[2][0], pred[0], offset, "")

                # Swap numpy buffer — pipeline graph stays intact
                refresh_train_data()

            # Train one epoch (native for-loop, graph-mode train_step)
            model.train_epoch(train_dataset=train_ds)

            epoch_loss = float(model.train_loss.result())
            print(f"Epoch {epoch + 1:>4d}  loss: {epoch_loss:.6f}")
            mlflow.log_metric("train_loss", epoch_loss, step=epoch)


fit(epochs)

2026/04/20 15:53:00 INFO mlflow.tracking.fluent: Experiment with name 'p_unet_320' does not exist. Creating a new experiment.


Creating new Data Points ...
It took 73 seconds
Creating new Data Points ...
It took 114 seconds


2026-04-20 15:56:13.260612: W tensorflow/core/framework/op_kernel.cc:1857] OP_REQUIRES failed at strided_slice_op.cc:117 : INVALID_ARGUMENT: Expected begin, end, and strides to be 1D equal size tensors, but got shapes [4], [1], and [1] instead.
2026-04-20 15:56:13.260663: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected begin, end, and strides to be 1D equal size tensors, but got shapes [4], [1], and [1] instead.
2026-04-20 15:56:13.265272: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: output shape must have two elements[0]
	 [[{{node ImageProjectiveTransformV3}}]]
2026-04-20 15:56:13.291043: W tensorflow/core/framework/op_kernel.cc:1844] UNKNOWN: InvalidArgumentError: Exception encountered when calling RandomRotation.call().

{{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:GPU:0}} Expected begin, end, and stri

UnknownError: {{function_node __wrapped__IteratorGetNext_output_types_4_device_/job:localhost/replica:0/task:0/device:CPU:0}} InvalidArgumentError: Exception encountered when calling RandomRotation.call().

[1m{{function_node __wrapped__ImageProjectiveTransformV3_device_/job:localhost/replica:0/task:0/device:GPU:0}} output shape must have two elements[0]
	 [[{{node ImageProjectiveTransformV3}}]] [Op:ImageProjectiveTransformV3] name: [0m

Arguments received by RandomRotation.call():
  • data=tf.Tensor(shape=(1, 128, 128, 4), dtype=float32)
  • training=True
Traceback (most recent call last):

  File "/software/anaconda3/envs/machauer/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 267, in __call__
    return func(device, token, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/software/anaconda3/envs/machauer/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 145, in __call__
    outputs = self._call(device, args)
              ^^^^^^^^^